In [ ]:
# stdlib pathlib (filesystem paths)
from pathlib import Path

# src/utils/pathing.py
from src.utils.pathing import ensure_repo_root_on_sys_path  # src/utils/pathing.py

ensure_repo_root_on_sys_path(Path.cwd())

# 16. Supervised Learning: Support Vector Machine (SVM)

## Algorithm Category
**Type**: Supervised Learning - Classification/Regression  
**Complexity**: Medium-High  
**Use Case**: Maximum margin classification with kernel methods

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the mathematical foundation of SVMs and maximum margin principle
- Implement SVM for classification using scikit-learn
- Understand kernel functions and their role in non-linear classification
- Tune hyperparameters (C, gamma, kernel type)
- Visualize decision boundaries and support vectors
- Apply SVM to real-world classification problems

## Historical Context

Support Vector Machines were developed in the 1990s, building on statistical learning theory:
- Vapnik and Chervonenkis (1960s-1970s): Statistical learning theory
- Boser, Guyon, Vapnik (1992): Kernel trick for non-linear SVMs
- Cortes and Vapnik (1995): Soft margin SVM

**Key Papers/References:**
- Cortes, C. & Vapnik, V. (1995). "Support-vector networks"
- Vapnik, V. (1998). "Statistical Learning Theory"
- Boser, B.E., et al. (1992). "A training algorithm for optimal margin classifiers"

## When to Use Support Vector Machines

SVMs are appropriate when:
- You need a robust classifier with good generalization
- Data has clear margin of separation
- High-dimensional data (works well with many features)
- Non-linear relationships (using kernel trick)
- Small to medium-sized datasets
- Memory efficiency is important (uses only support vectors)

## Theory & Mechanics

### Mathematical Foundation

SVM finds the optimal hyperplane that maximizes the margin between classes.

**Hard Margin SVM (linearly separable):**
- Maximize margin: $\frac{2}{||w||}$
- Subject to: $y_i(w \cdot x_i + b) \geq 1$ for all $i$

**Soft Margin SVM (not perfectly separable):**
- Minimize: $\frac{1}{2}||w||^2 + C\sum_{i=1}^{n}\xi_i$
- Subject to: $y_i(w \cdot x_i + b) \geq 1 - \xi_i$, $\xi_i \geq 0$

**Kernel Trick:**
- Maps data to higher-dimensional space: $\phi(x)$
- Kernel function: $K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$
- Common kernels: Linear, Polynomial, RBF (Gaussian), Sigmoid

**Dual Form (Lagrange multipliers):**
$$\max_{\alpha} \sum_{i=1}^{n}\alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j K(x_i, x_j)$$

### How It Works

1. **Find Support Vectors**: Data points closest to decision boundary
2. **Maximize Margin**: Find hyperplane with maximum distance to nearest points
3. **Kernel Transformation**: Map to higher dimension if needed (kernel trick)
4. **Prediction**: Classify based on which side of hyperplane

### Key Hyperparameters

- **C**: Regularization parameter (controls trade-off between margin and misclassification)
- **kernel**: Type of kernel ('linear', 'poly', 'rbf', 'sigmoid')
- **gamma**: Kernel coefficient for 'rbf', 'poly', 'sigmoid' (defines influence of single training example)
- **degree**: Degree for polynomial kernel

### Limitations

- Memory intensive for large datasets (O(n²) or O(n³))
- Slow training on large datasets
- Sensitive to feature scaling
- Difficult to interpret (black box model)
- Requires careful hyperparameter tuning


## Implementation

Let's implement SVM for classification with different kernels.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_breast_cancer,  # Breast cancer classification dataset
    make_classification,  # Generate synthetic classification data
    make_circles  # Generate circular classification data (for non-linear examples)
)
from sklearn.svm import SVC  # Support Vector Classifier (SVM for classification)
from sklearn.model_selection import (
    train_test_split,  # Split data into train/test sets
    cross_val_score,  # Cross-validation scoring
    GridSearchCV  # Hyperparameter tuning
)
from sklearn.metrics import (
    accuracy_score,  # Calculate accuracy
    classification_report  # Detailed classification metrics
)
from sklearn.preprocessing import StandardScaler  # Feature scaling

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import split_data, evaluate_classifier  # Supervised learning utilities
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix,  # Visualize confusion matrix
    plot_roc_curve,  # Plot ROC curve
    trace_support_vectors  # Visualize support vectors
)
from src.processing.preprocessing import scale_features  # Normalize features
from src.utils.benchmarking import benchmark_model_training  # Measure training time
from src.utils.traceability import save_traceability_data  # Save model information
from src.utils.validation import (
    validate_model_output,  # Check if predictions are valid
    check_cross_validation_stability  # Check CV stability
)

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Breast Cancer Classification
# ============================================

# load_breast_cancer() loads the Breast Cancer Wisconsin dataset from scikit-learn
# This is a binary classification problem: predict if tumor is malignant (1) or benign (0)
cancer = load_breast_cancer()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Medical measurements of breast tumors
# cancer.data contains feature values (569 samples × 30 features)
# We convert to DataFrame for easier manipulation
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
# Features include: mean radius, mean texture, mean perimeter, mean area, etc. (30 total)

# y = Target (output): Tumor type (what we want to predict)
# cancer.target contains class labels (0 = benign, 1 = malignant)
y = pd.Series(cancer.target, name='Target')
# 0 = benign (non-cancerous), 1 = malignant (cancerous)

print(f"Dataset Shape: {X.shape}")  # Output: (569, 30) - 569 patients, 30 features
print(f"Classes: {cancer.target_names}")  # Output: ['malignant' 'benign']

# ============================================
# FEATURE SCALING: Critical for SVM
# ============================================

# SVM is VERY sensitive to feature scaling!
# Why? SVM tries to maximize the margin, and features on different scales distort distances
# Without scaling, features with larger values dominate the distance calculations
# This can cause poor performance or slow convergence

# scale_features() normalizes features to have mean=0 and std=1
# fit=True means "learn scaling from this data" (use for training data)
X_scaled, scaler = scale_features(X, fit=True)
# X_scaled: Features normalized (mean=0, std=1 for each column)
# scaler: The scaling object (needed to scale test data with same transformation)

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# CRITICAL: Split AFTER scaling to prevent data leakage
# If we split first, test data statistics could influence scaling

# split_data() randomly splits data into training (80%) and test (20%) sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state=42 ensures same split every time (reproducibility)
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)
# X_train: 80% of samples for training (scaled features)
# X_test: 20% of samples for testing (will be scaled using same scaler later)
# y_train: Target values for training samples
# y_test: Target values for test samples (ground truth)


In [ ]:
# ============================================
# MODEL CREATION: Support Vector Machine (SVM)
# ============================================

# Create an SVC (Support Vector Classifier) model
# This model will find the optimal hyperplane that maximizes the margin between classes

# SVC parameters:
# kernel='rbf': Radial Basis Function (Gaussian) kernel
#   - RBF kernel maps data to infinite-dimensional space
#   - Allows SVM to find non-linear decision boundaries
#   - Good default choice for most problems
#   - Other options: 'linear', 'poly', 'sigmoid'
#
# C=1.0: Regularization parameter
#   - Controls trade-off between margin size and classification errors
#   - Smaller C = larger margin, more misclassifications allowed (softer margin)
#   - Larger C = smaller margin, fewer misclassifications (harder margin)
#   - Default is 1.0
#
# gamma='scale': Kernel coefficient for RBF
#   - 'scale' means: 1 / (n_features * X.var())
#   - Controls influence of single training example
#   - Smaller gamma = larger influence radius (smoother decision boundary)
#   - Larger gamma = smaller influence radius (more complex decision boundary)
#   - Other options: 'auto' or a float value
#
# random_state=42: Ensures reproducible results
#
# probability=True: Enable probability estimates
#   - Allows .predict_proba() to return class probabilities
#   - Uses cross-validation internally (slower but more accurate probabilities)
model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42, probability=True)

# ============================================
# MODEL TRAINING: Learning from Data
# ============================================

# .fit() trains the SVM on training data
# The algorithm:
# 1. Finds support vectors (data points closest to decision boundary)
# 2. Maximizes margin (distance between decision boundary and nearest points)
# 3. Uses kernel trick to handle non-linear relationships
# Internally uses quadratic programming optimization
model.fit(X_train, y_train)  # Train the model

print("Model trained successfully!")  # Confirm training completed

print("Model trained successfully!")  # Confirm training completed

# ============================================
# SUPPORT VECTORS: Understanding SVM's Key Concept
# ============================================

# Support vectors are the data points closest to the decision boundary
# They "support" the margin - if you remove them, the margin changes
# SVM only uses support vectors for prediction (not all training data!)
# This makes SVM memory-efficient (only stores support vectors)

print(f"Number of support vectors: {model.n_support_}")  # Array: support vectors per class
# For binary classification: [n_support_class_0, n_support_class_1]
# Example: [15, 20] means 15 support vectors for class 0, 20 for class 1

print(f"Support vectors per class: {model.n_support_}")
# Total support vectors = sum of n_support_
# Only these points are needed for prediction (much less than all training data!)

# ============================================
# MAKING PREDICTIONS
# ============================================

# .predict() makes class predictions using support vectors
# For each test sample:
# 1. Calculate distance to support vectors
# 2. Apply kernel function
# 3. Weight by support vector coefficients
# 4. Sum and apply sign function
# 5. Return predicted class
y_pred = model.predict(X_test)  # Class predictions (0 or 1)

# .predict_proba() returns probability estimates
# probability=True was set during model creation to enable this
y_pred_proba = model.predict_proba(X_test)[:, 1]
# predict_proba() returns probabilities for each class: [[P(class0), P(class1)], ...]
# [:, 1] extracts probability of class 1 (positive class, malignant)
# Range: 0 to 1 (higher = more confident it's malignant)

# ============================================
# EVALUATING MODEL PERFORMANCE
# ============================================

# Evaluate with helper function
results = evaluate_classifier(model, X_test, y_test)
# Returns dictionary with accuracy and other metrics

# Calculate detailed classification metrics
metrics = calculate_classification_metrics(y_test.values, y_pred, y_pred_proba)
# y_test.values: True labels
# y_pred: Class predictions
# y_pred_proba: Probability estimates (for ROC-AUC)
# Returns dictionary with: accuracy, precision, recall, F1, ROC-AUC

# ============================================
# DISPLAYING PERFORMANCE METRICS
# ============================================

print(f"\nClassification Metrics:")

# Accuracy: (correct predictions) / (total predictions)
# Range: 0 to 1. Higher is better
print(f"  Accuracy: {metrics['accuracy']:.3f}")  # Overall correctness

# Precision: (true positives) / (true positives + false positives)
# "Of all predicted positives, how many were actually positive?"
# Range: 0 to 1. Higher is better
print(f"  Precision: {metrics['precision']:.3f}")  # Precision of positive predictions

# Recall (Sensitivity): (true positives) / (true positives + false negatives)
# "Of all actual positives, how many did we catch?"
# Range: 0 to 1. Higher is better
print(f"  Recall: {metrics['recall']:.3f}")  # Ability to find all positives

# F1 Score: Harmonic mean of precision and recall
# F1 = 2 × (precision × recall) / (precision + recall)
# Range: 0 to 1. Higher is better. Balances precision and recall
print(f"  F1 Score: {metrics['f1_score']:.3f}")  # Balanced metric

# ROC-AUC: Area under the ROC curve
# Measures ability to distinguish between classes
# Range: 0 to 1. 1.0 = perfect, 0.5 = random, <0.5 = worse than random
if 'roc_auc' in metrics:
    print(f"  ROC-AUC: {metrics['roc_auc']:.3f}")  # Classification performance


## Validation & Testing

Let's validate our model and compare different kernels.


In [ ]:
# ============================================
# VALIDATION 1: Checking Model Output Validity
# ============================================

# validate_model_output() checks if predictions are valid
# task_type='classification' tells validator this is classification
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
# y_pred: Predictions from SVM
# y_test.values: True labels
# task_type='classification': Specify task type
# Returns dictionary with validation results

print("Model Output Validation:")
print(f"  Valid: {validation_result['valid']}")  # True if predictions are valid

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: Predictions must be valid
assert validation_result['valid'], "Invalid predictions!"
# If predictions are invalid (wrong shape, wrong values, etc.), stop execution

# Check 2: Accuracy must be better than random guessing
# For binary classification, random = 0.5 (50%)
assert metrics['accuracy'] > 0.5, "Accuracy should be better than random!"
# If accuracy ≤ 0.5, model is no better than guessing

print("\n✓ Basic validation checks passed")  # All checks passed!


In [ ]:
# Validation 2: Compare different kernels
kernels = ['linear', 'poly', 'rbf', 'sigmoid']
kernel_results = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, C=1.0, random_state=42, probability=True)
    svm.fit(X_train, y_train)
    pred = svm.predict(X_test)
    acc = accuracy_score(y_test, pred)
    kernel_results[kernel] = {
        'accuracy': acc,
        'n_support_vectors': svm.n_support_.sum()
    }
    print(f"{kernel.upper()} kernel: Accuracy={acc:.3f}, Support Vectors={svm.n_support_.sum()}")

best_kernel = max(kernel_results, key=lambda k: kernel_results[k]['accuracy'])
print(f"\nBest kernel: {best_kernel} with accuracy {kernel_results[best_kernel]['accuracy']:.3f}")


In [ ]:
# Validation 3: Cross-validation
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")
print("\n✓ Cross-validation checks passed")


## Traceability

Let's trace support vectors and visualize decision boundaries.


In [ ]:
# Trace support vectors
sv_info = trace_support_vectors(model, X_train.values, y_train.values)
print("Support Vector Information:")
print(f"  Total support vectors: {sv_info['n_support_vectors']}")
print(f"  Support vector indices: {sv_info['support_indices'][:10]}...")  # Show first 10

# Visualize support vectors (using first 2 features for 2D plot)
plt.figure(figsize=(12, 5))

# Plot 1: Support vectors in feature space
plt.subplot(1, 2, 1)
plt.scatter(X_train.iloc[:, 0], X_train.iloc[:, 1], c=y_train, cmap='viridis', alpha=0.3)
plt.scatter(X_train.iloc[sv_info['support_indices'], 0], 
           X_train.iloc[sv_info['support_indices'], 1],
           s=200, facecolors='none', edgecolors='red', linewidths=2, label='Support Vectors')
plt.xlabel(cancer.feature_names[0])
plt.ylabel(cancer.feature_names[1])
plt.title('Support Vectors (First 2 Features)')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: ROC curve
plt.subplot(1, 2, 2)
roc_auc, fpr, tpr = plot_roc_curve(y_test.values, y_pred_proba, 
                                   title="SVM ROC Curve")
print(f"\nROC-AUC Score: {roc_auc:.3f}")

plt.tight_layout()
plt.show()


## Visualization: Non-linear Classification

Let's visualize how different kernels handle non-linear data.


In [ ]:
# Create non-linear dataset (circles)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=42)
X_circles_train, X_circles_test, y_circles_train, y_circles_test = train_test_split(
    X_circles, y_circles, test_size=0.2, random_state=42
)

# Train SVMs with different kernels
kernels_to_test = ['linear', 'poly', 'rbf']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, kernel in enumerate(kernels_to_test):
    svm = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42)
    svm.fit(X_circles_train, y_circles_train)
    
    # Create mesh for decision boundary
    h = 0.02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                        np.arange(y_min, y_max, h))
    
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    axes[idx].contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    axes[idx].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', edgecolors='black')
    axes[idx].set_title(f'{kernel.upper()} Kernel')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('SVM Decision Boundaries with Different Kernels')
plt.tight_layout()
plt.show()


## Real-World Application

Let's tune hyperparameters using GridSearchCV.


In [ ]:
# Hyperparameter tuning
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'poly']
}

# Use smaller grid for faster execution
grid_search = GridSearchCV(
    SVC(random_state=42, probability=True),
    param_grid,
    cv=3,  # Reduced folds for speed
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("\nBest Hyperparameters:")
print(grid_search.best_params_)
print(f"Best CV Accuracy: {grid_search.best_score_:.3f}")

# Evaluate best model
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, best_pred)
print(f"Test Accuracy with Best Model: {best_accuracy:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **SVM Basics**
   - Finds optimal hyperplane with maximum margin
   - Uses support vectors (critical data points) for decision boundary
   - Can handle non-linear data through kernel trick

2. **Kernels**
   - **Linear**: For linearly separable data
   - **Polynomial**: For polynomial decision boundaries
   - **RBF (Gaussian)**: Most versatile, handles complex non-linear patterns
   - **Sigmoid**: Similar to neural network activation

3. **Hyperparameters**
   - **C**: Controls regularization (higher = less regularization, tighter fit)
   - **gamma**: Controls influence of single training example (higher = more complex boundaries)
   - **kernel**: Type of kernel function

4. **Best Practices**
   - Always scale features (SVM is sensitive to feature scales)
   - Start with RBF kernel for non-linear problems
   - Use cross-validation for hyperparameter tuning
   - Consider computational cost for large datasets

### When to Use Support Vector Machines

✅ **Good for:**
- High-dimensional data
- Clear margin of separation
- Non-linear relationships (with kernels)
- Small to medium datasets
- When you need a robust classifier

❌ **Not ideal for:**
- Very large datasets (memory intensive)
- Noisy data with overlapping classes
- When interpretability is crucial
- When probability estimates are needed (can be slow)
- Real-time predictions on large datasets

### Next Steps

- Try **SVR (Support Vector Regression)** for regression tasks
- Explore **Nu-SVM** for different parameterization
- Consider **LinearSVC** for faster linear SVM
- Compare with **Logistic Regression** for similar use cases
